# 03 streaming + stopping

目标：理解流式输出和停止条件。streaming 不减少总计算量，但能显著改善用户感知延迟。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


In [ ]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


## 1. 加载组件和自定义停止条件

`StoppingCriteria` 可以在生成循环里检查最新 token，满足条件就停止。


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
    StoppingCriteria,
    StoppingCriteriaList,
    TextStreamer,
)


class StopOnTokenIds(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = set(stop_token_ids)

    def __call__(self, input_ids, scores, **kwargs):
        if not self.stop_token_ids:
            return False
        return input_ids[0, -1].item() in self.stop_token_ids


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)


## 2. 准备 prompt 和生成配置


In [ ]:
messages = [
    {"role": "system", "content": "你是一个大模型部署工程师。"},
    {"role": "user", "content": "用要点解释：为什么 streaming 能改善体验？"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

generation_config = GenerationConfig(
    max_new_tokens=160,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)


## 3. 流式输出

`TextStreamer` 会在生成过程中逐步打印新增文本。


In [ ]:
streamer = TextStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

stopping_criteria = StoppingCriteriaList(
    [StopOnTokenIds([tokenizer.eos_token_id])]
)

model.generate(
    **inputs,
    generation_config=generation_config,
    streamer=streamer,
    stopping_criteria=stopping_criteria,
)
